# Recommendation bench

Compare recommendation sources on the `albums_3plus.csv` benchmark set.

Start by loading `recommendations_all_sources.parquet` and filtering it to albums in the bench list.

In [6]:
import re
import unicodedata
from pathlib import Path

import pandas as pd

DATA_DIR = Path("../datasets")
RECOMMENDATIONS_PATH = DATA_DIR / "recommendations_all_sources.parquet"
BENCH_ALBUMS_PATH = DATA_DIR / "albums_2plus.csv"

In [7]:
def normalize(s):
    if pd.isna(s):
        return ""
    s = str(s).strip().lower()
    for ch in ("\u2019", "\u2018", "\u201b", "\u2032", "`", "\u00b4"):
        s = s.replace(ch, "'")
    s = s.replace("\u2026", "...").replace("\u00b7", " ")
    s = unicodedata.normalize("NFKD", s)
    s = "".join(c for c in s if not unicodedata.combining(c))
    s = re.sub(r"\s+", " ", s)
    return s


def pair_key(artist, album):
    return f"{normalize(artist)}::{normalize(album)}"

In [8]:
recs_all = pd.read_parquet(RECOMMENDATIONS_PATH)
bench_albums = pd.read_csv(BENCH_ALBUMS_PATH)

print(f"All recommendations: {len(recs_all):,} rows")
print(f"Bench albums: {len(bench_albums):,}")
recs_all.head(2)

All recommendations: 481,890 rows
Bench albums: 2,498


,query_review_id,query_artist,query_album,query_dataset,rank,rec_review_id,rec_artist,rec_album,rec_dataset,rec_score
0,60d4e2a1-53be-4ca9-9683-31db6b7cbe46,Astrid Hadad,La Pluma o La Espada,critique_brainz,1,14a0e5f7-c20d-4633-8289-1fb42d609e6c,Lila Downs,La cantina: “Entre copa y copa…”,critique_brainz,0.7901
1,60d4e2a1-53be-4ca9-9683-31db6b7cbe46,Astrid Hadad,La Pluma o La Espada,critique_brainz,2,7e6830cc-c770-4ecf-a0e4-0259484c9a66,"The Harp Consort, Andrew Lawrence‐King",Missa Mexicana,critique_brainz,0.7850


In [9]:
bench_albums["pair_key"] = [
    pair_key(artist, album) for artist, album in zip(bench_albums["artist"], bench_albums["album"])
]
bench_keys = set(bench_albums["pair_key"])

recs_all["pair_key"] = [
    pair_key(artist, album)
    for artist, album in zip(recs_all["query_artist"], recs_all["query_album"])
]

bench_recs = recs_all[recs_all["pair_key"].isin(bench_keys)].copy()
bench_recs = bench_recs.merge(
    bench_albums[["album_id", "pair_key", "artist", "album", "review_count"]],
    on="pair_key",
    how="left",
)

matched_albums = bench_albums["pair_key"].isin(bench_recs["pair_key"].unique()).sum()
print(f"Matched bench albums: {matched_albums} / {len(bench_albums)}")
print(f"Filtered recommendations: {len(bench_recs):,} rows")
print(f"Unique bench query albums in recs: {bench_recs['pair_key'].nunique()}")
print("\nRecommendation sources:")
print(bench_recs["rec_dataset"].value_counts())

Matched bench albums: 2498 / 2498
Filtered recommendations: 50,480 rows
Unique bench query albums in recs: 2498

Recommendation sources:
rec_dataset
pitchfork           32832
critique_brainz     13176
resident_advisor     4472
Name: count, dtype: int64


In [10]:
# One row per bench album with recommendation counts by source
(
    bench_recs.groupby(["album_id", "artist", "album", "rec_dataset"], as_index=False)
    .size()
    .rename(columns={"size": "n_recs"})
    .sort_values(["artist", "album", "rec_dataset"])
    .head(20)
)

,album_id,artist,album,rec_dataset,n_recs
0,3,!!!,Myth Takes,critique_brainz,2
1,3,!!!,Myth Takes,pitchfork,16
2,3,!!!,Myth Takes,resident_advisor,2
3,5,!!!,"Strange Weather, Isn't It?",critique_brainz,1
4,5,!!!,"Strange Weather, Isn't It?",pitchfork,18
5,5,!!!,"Strange Weather, Isn't It?",resident_advisor,1
6,57,1990s,Cookies,critique_brainz,6
7,57,1990s,Cookies,pitchfork,14
8,58,1990s,Kicks,critique_brainz,5
9,58,1990s,Kicks,pitchfork,15


In [11]:
unmatched = bench_albums.loc[~bench_albums["pair_key"].isin(bench_recs["pair_key"].unique())]
if unmatched.empty:
    print("All bench albums matched.")
else:
    print(f"Unmatched bench albums: {len(unmatched)}")
    unmatched[["album_id", "artist", "album", "review_count"]]

All bench albums matched.
